# 06 · Construcción de una aplicación de analítica financiera

**Financial Analytics Lab · Python · Streamlit · pandas · yfinance**

**Pregunta orientadora:** ¿cómo convertir los análisis desarrollados en notebooks en una herramienta interactiva que permita consultar activos, interpretar indicadores y comunicar sus limitaciones?

Este módulo reúne los aprendizajes anteriores en **un solo proyecto incremental**. La primera versión (S27) debe funcionar con un activo; la ampliación (S33) incorpora comparación de activos, ponderaciones y riesgo; la defensa individual (S34) permite demostrar comprensión del código y los resultados.

**Nota académica:** S27, S33 y S34 son hitos diferenciados de un mismo proyecto. Su aplicación como equivalencias evaluativas requiere autorización institucional.

**Objetivos:** comprender la arquitectura de Streamlit, organizar un proyecto Python, conectar controles con cálculos, validar entradas, ejecutar una app y documentar sus resultados.

## 0 · ¿En qué se diferencia una aplicación de un notebook?

En un notebook ejecutamos celdas de forma progresiva. En una aplicación Streamlit, el archivo `app.py` describe una interfaz y se vuelve a ejecutar cuando el usuario cambia ciertos controles. Por eso debemos separar las responsabilidades:

**Entrada → validación → consulta → preparación → indicadores → gráficos → interpretación.**

El archivo `requirements.txt` registra las dependencias; `README.md` explica cómo utilizar el proyecto. Los datos no deben incluir credenciales privadas.

**Pregunta:** ¿qué puede ocurrir si el usuario elige una fecha final anterior a la inicial?

## 1 · Conocer Streamlit

Streamlit es un marco de Python para crear aplicaciones interactivas. Sus componentes incluyen `st.title` para títulos, `st.selectbox` para elegir opciones, `st.date_input` para fechas, `st.metric` para indicadores y `st.pyplot` para mostrar gráficos.

**Documentación oficial:** https://docs.streamlit.io/ .

En este notebook examinaremos piezas del programa y verificaremos los cálculos. Para **ejecutar la interfaz completa**, se recomienda descargar o clonar la carpeta `06_financial_app` y utilizar una terminal; el entorno de Google Colab no ofrece por sí solo un enlace público estable para una aplicación Streamlit.

## 2 · Estructura del proyecto

```text
06_financial_app/
├── app.py                  # Versión inicial: un activo (S27)
├── app_final.py            # Versión ampliada: varios activos (S33)
├── requirements.txt        # Bibliotecas requeridas
├── .streamlit/config.toml  # Apariencia
├── 06_guia_streamlit.ipynb # Esta guía
└── README.md               # Instrucciones y evidencias
```

En un computador con Python instalado, abre una terminal dentro de `06_financial_app` y ejecuta:

```bash
python -m pip install -r requirements.txt
python -m streamlit run app.py
```

Para la versión ampliada:

```bash
python -m streamlit run app_final.py
```

**No ejecutes esos comandos como si fueran celdas de Python**: son instrucciones de terminal.

## 3 · Probar la lógica antes de construir la interfaz

Una app confiable debe separar la lógica de cálculo de la visualización. Vamos a trabajar con una serie ficticia para comprobar el rendimiento acumulado, la volatilidad y el drawdown sin depender de internet.

In [ ]:
import pandas as pd
import numpy as np

precios_ejemplo = pd.Series([100.0, 105.0, 102.0, 110.0], name="Cierre")
retornos = precios_ejemplo.pct_change(fill_method=None).dropna()
rendimiento_acumulado = precios_ejemplo.iloc[-1] / precios_ejemplo.iloc[0] - 1
volatilidad = retornos.std(ddof=1)
indice = precios_ejemplo / precios_ejemplo.iloc[0]
drawdown = indice / indice.cummax() - 1

print(f"Rendimiento acumulado: {rendimiento_acumulado:.2%}")
print(f"Volatilidad por observación: {volatilidad:.2%}")
print(f"Máximo drawdown: {drawdown.min():.2%}")
assert np.isclose(rendimiento_acumulado, 0.10)
assert (drawdown <= 0).all()

**Interpretación:** el precio pasa de 100 a 110, equivalente a un aumento acumulado de 10 %. La volatilidad resume la dispersión de los rendimientos observados; el drawdown compara cada valor con su máximo anterior.

**Tu práctica 1:** modifica la serie para que el último precio sea 95. Comprueba el rendimiento acumulado y explica por qué cambia el resultado.

In [ ]:
# TU PRÁCTICA 1
# Crea otra serie y contrasta sus indicadores.

## 4 · Versión inicial (S27): una aplicación funcional

La primera versión tiene un alcance deliberadamente acotado. El usuario selecciona un activo y un rango de fechas. El programa consulta cierres ajustados, comprueba que existan observaciones suficientes, calcula rendimiento y volatilidad, presenta un gráfico y explica el significado de los resultados.

**Archivos:** `app.py`, `requirements.txt` y `README.md`.

**Comprueba:**

1. ¿Qué ocurre si las fechas son inválidas?
2. ¿Qué sucede cuando no hay datos?
3. ¿Qué significa cada indicador y cuáles son sus unidades?
4. ¿La interpretación distingue pasado y futuro?
5. ¿La fuente y las decisiones de limpieza quedan documentadas?

**Tu práctica 2:** abre `app.py` en GitHub, identifica dónde se realiza cada una de estas cinco acciones y explica por qué se utiliza `st.cache_data`.

In [ ]:
# TU PRÁCTICA 2
# Documenta en una celda Markdown los componentes de app.py.

## 5 · Introducir interacción y validación

Los controles de Streamlit no son adornos: determinan las entradas del análisis. Una fecha inválida o un ticker inexistente debe producir un mensaje comprensible, no un resultado inventado.

El siguiente fragmento muestra la idea central; se ejecuta dentro de `app.py`, no directamente en este notebook:

```python
activo = st.sidebar.selectbox('Activo', ['MSFT', 'AAPL', 'JPM'])
fecha_inicio = st.sidebar.date_input('Inicio')
fecha_fin = st.sidebar.date_input('Fin')
if fecha_inicio >= fecha_fin:
    st.error('La fecha inicial debe ser anterior a la final.')
    st.stop()
```

**Tu práctica 3:** añade un activo disponible en Yahoo Finance al diccionario de opciones de `app.py`. Explica qué validación harías si la fuente no devolviera observaciones.

## 6 · Versión ampliada (S33): comparar activos y portafolios

La versión `app_final.py` amplía el mismo proyecto. Permite seleccionar entre dos y cuatro activos, definir ponderaciones, consultar fechas comunes y calcular:

- Rendimiento acumulado del portafolio.
- Volatilidad histórica por observación.
- Matriz de correlaciones.
- Máximo drawdown.
- Trayectorias de precios normalizados y valor relativo del portafolio.

**Supuesto decisivo:** los pesos se mantienen constantes mediante reequilibrio hipotético en cada observación. Sin ese supuesto, la combinación de rendimientos requiere seguir las cantidades y los cambios de peso a lo largo del tiempo.

**Tu práctica 4:** explica por qué los pesos deben sumar 100 % y qué representa una correlación positiva elevada entre dos activos.

## 7 · Verificar el rendimiento de un portafolio ponderado

El rendimiento por observación de un portafolio con pesos constantes se calcula como suma ponderada. La volatilidad depende de la matriz de covarianzas:

\[r_{p,t}=\sum_i w_i r_{i,t}\]

\[\sigma_p=\sqrt{\mathbf{w}^T\mathbf{\Sigma}\mathbf{w}}\]

Comprobaremos ambas relaciones con una muestra hipotética.

In [ ]:
muestra = pd.DataFrame({
    "ACT_A": [0.02, -0.01, 0.03, 0.00],
    "ACT_B": [0.01, 0.02, -0.01, 0.01],
    "ACT_C": [-0.01, 0.00, 0.01, 0.02]
})
pesos = pd.Series({"ACT_A": 0.4, "ACT_B": 0.35, "ACT_C": 0.25})
r_port = muestra.dot(pesos)
vol_directa = r_port.std(ddof=1)
vol_matriz = np.sqrt(float(pesos.to_numpy() @ muestra.cov().to_numpy() @ pesos.to_numpy()))
assert np.isclose(pesos.sum(), 1)
assert np.isclose(vol_directa, vol_matriz)
print("Rendimientos del portafolio:", r_port.round(4).tolist())
print("Volatilidad comprobada:", round(vol_directa, 6))

**Tu práctica 5:** modifica los pesos y verifica de nuevo la volatilidad. Explica por qué la suma ponderada de desviaciones estándar individuales no es, en general, la volatilidad correcta del portafolio.

In [ ]:
# TU PRÁCTICA 5
# Define nuevos pesos y verifica la fórmula.

## 8 · Evidencias diferenciadas del mismo proyecto

**S27 — Primera versión funcional:** ejecuta `app.py`, presenta una consulta válida, muestra un indicador de rendimiento, uno de volatilidad, un gráfico y una interpretación. Conserva código y documentación.

**S33 — Versión ampliada:** ejecuta `app_final.py`, selecciona varios activos, documenta los pesos y comprueba al menos un cálculo. Explica las diferencias respecto a la primera versión.

**S34 — Defensa individual:** explica la estructura del programa, las decisiones de limpieza, las fórmulas, las limitaciones y una modificación sencilla del código. La modalidad de defensa debe acordarse con la institución.

**Importante:** estos son hitos propuestos de un mismo proyecto, no una modificación automática del esquema de evaluación del curso.

## 9 · Reto final: documentar y explicar tu aplicación

Elabora una entrega individual con:

1. Una pregunta de análisis financiero y la descripción del usuario de la aplicación.
2. Repositorio organizado con código, dependencias y README.
3. Evidencia de funcionamiento de la primera versión y de la ampliación.
4. Explicación de entradas, validaciones, cálculos, gráficos e interpretación.
5. Una comprobación independiente del rendimiento y la volatilidad.
6. Un ejemplo de error de entrada y cómo lo gestiona la aplicación.
7. Un análisis escrito de **250 a 350 palabras** sobre hallazgos, decisiones técnicas y limitaciones.
8. Una bitácora breve de cambios entre las versiones, indicando apoyos externos o IA utilizados.

**Criterio de calidad:** una aplicación visualmente atractiva no sustituye la corrección de los cálculos ni la comprensión de sus supuestos.

## 10 · Autoevaluación y continuidad

- [ ] Distingo la lógica de cálculo de los componentes visuales.
- [ ] Sé ejecutar una aplicación Streamlit desde una terminal.
- [ ] Compruebo las entradas y manejo datos faltantes.
- [ ] Interpreto rendimiento, volatilidad, correlación y drawdown.
- [ ] Comprendo el supuesto de reequilibrio de pesos.
- [ ] Puedo explicar y modificar el código de forma autónoma.
- [ ] Documento el procedimiento, la fuente y las limitaciones.

**Documentación:** [Streamlit](https://docs.streamlit.io/) · [pandas](https://pandas.pydata.org/docs/) · [yfinance](https://ranaroussi.github.io/yfinance/) · [GitHub](https://docs.github.com/es).

**Cierre del recorrido:** el resultado es una herramienta analítica reproducible, no una predicción de precios ni una recomendación de inversión.